# 00 · EDA

What the gold answers look like (length, style) and how the data is spread over countries and categories.
Uses the `qa` jsonl files only, no images.

- dev has no `country` / `category`, so those come from train and devtest.
- To get train / devtest jsonl without the image zips: `python download_data.py train devtest --task qa --no-media`

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from common import RESULTS_DIR, load_split, split_path

pd.set_option("display.max_colwidth", 200)

frames = []
for split in ("train", "dev", "devtest"):
    if split_path("qa", split).exists():
        frames.append(pd.DataFrame(load_split("qa", split)))
    else:
        print(f"skip {split}: {split_path('qa', split)} not found")
df = pd.concat(frames, ignore_index=True)
print(df.groupby("split").size())
print("columns:", list(df.columns))

## 1. Length of answers and questions

In [ ]:
def n_words(s): return len(s.split())
def n_sents(s): return max(1, len(re.findall(r"[.!?](?:\s|$)", s.strip())))

df["a_words"] = df["answer"].map(n_words)
df["q_words"] = df["question"].map(n_words)
df["a_sents"] = df["answer"].map(n_sents)

def describe(col):
    return df.groupby("split")[col].describe(percentiles=[.05, .25, .5, .75, .95]).round(1)

print("Answer length, words"); display(describe("a_words"))
print("Question length, words"); display(describe("q_words"))
print("Sentences per answer"); display(pd.crosstab(df["split"], df["a_sents"].clip(upper=3), normalize="index").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for split, g in df.groupby("split"):
    axes[0].hist(g["a_words"], bins=range(0, 70, 2), alpha=.5, density=True, label=split)
    axes[1].hist(g["q_words"], bins=range(0, 30), alpha=.5, density=True, label=split)
axes[0].set(title="Answer length", xlabel="words")
axes[1].set(title="Question length", xlabel="words")
axes[0].legend()
plt.tight_layout(); plt.show()

print("corr(question words, answer words):", df[["q_words", "a_words"]].corr().iloc[0, 1].round(2))

## 2. Answer style

In [ ]:
STOP = {"the", "a", "an", "of", "in", "is", "are", "what", "which", "this", "these", "that", "and", "to",
        "for", "on", "with", "does", "do", "image", "shown", "picture", "photo", "by", "be", "can"}
def content(s): return set(re.findall(r"[a-z']+", s.lower())) - STOP

HEDGE = r"\b(?:likely|possibly|probably|may|might|suggests?|appears?|seems?)\b"
IMAGE_REF = r"\b(?:the image|this image|the photo|the picture|depicted|shown|visible|in the background)\b"

a = df["answer"].str.strip()
style = pd.DataFrame({
    "split": df["split"],
    "starts_with_The": a.str.startswith("The "),
    "ends_with_period": a.str.endswith("."),
    "full_sentence": a.str.contains(r"\b(?:is|are|was|were|has|have|shows?|depicts?|represents?)\b", case=False),
    "mentions_image": a.str.contains(IMAGE_REF, case=False),
    "hedging": a.str.contains(HEDGE, case=False),
    "yes_no": a.str.lower().str.match(r"^(yes|no)\b"),
})
display(style.groupby("split").mean().round(3))

df["q_overlap"] = [len(content(x) & content(q)) / max(1, len(content(q))) for x, q in zip(df["answer"], df["question"])]
print("Share of question content words repeated in the answer:", df.groupby("split")["q_overlap"].mean().round(2).to_dict())

In [ ]:
print("Most common answer openings (first 2 words):")
display(df["answer"].str.split().str[:2].str.join(" ").value_counts().head(15).to_frame("count"))
print("Most common question openings (first 2 words):")
display(df["question"].str.split().str[:2].str.join(" ").value_counts().head(15).to_frame("count"))

In [ ]:
dev = df[df["split"] == "dev"] if (df["split"] == "dev").any() else df
for name, part in [("shortest", dev.nsmallest(5, "a_words")), ("median", dev[dev["a_words"] == dev["a_words"].median()].head(3)), ("longest", dev.nlargest(3, "a_words"))]:
    print(f"--- {name}")
    for _, r in part.iterrows():
        print(f"[{r.a_words}w] Q: {r.question}\n      A: {r.answer}\n")

## 3. Countries, categories, subcategories

Only where the labels exist (train, devtest). For dev there is a rough proxy below: country names and cities mentioned in the question or answer.

In [ ]:
labeled = df[df["country"].notna()] if "country" in df else df.iloc[0:0]
if labeled.empty:
    print("No country / category labels loaded. Download train or devtest jsonl (see top of the notebook).")
else:
    for col in ("country", "category"):
        t = pd.crosstab(labeled[col], labeled["split"])
        t["share"] = (t.sum(axis=1) / t.values.sum()).round(3)
        display(t.sort_values("share", ascending=False))
    print("subcategories:", labeled["subcategory"].nunique())
    display(labeled["subcategory"].value_counts().head(20).to_frame("count"))

In [ ]:
if not labeled.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labeled["country"].value_counts().sort_values().plot.barh(ax=axes[0], title="Examples per country")
    labeled["category"].value_counts().sort_values().plot.barh(ax=axes[1], title="Examples per category")
    plt.tight_layout(); plt.show()

    print("Answer length (words) by category")
    display(labeled.groupby("category")["a_words"].agg(["count", "mean", "median"]).round(1).sort_values("mean"))
    print("Category x country (counts)")
    display(pd.crosstab(labeled["category"], labeled["country"]))

In [ ]:
# Dev has no labels: rough country guess from names in the question / answer text.
PLACES = {
    "Egypt": ["egypt", "cairo", "giza", "luxor", "alexandria", "aswan"],
    "UAE": ["uae", "emirat", "dubai", "abu dhabi", "sharjah"],
    "Saudi Arabia": ["saudi", "riyadh", "mecca", "makkah", "medina", "jeddah"],
    "Qatar": ["qatar", "doha"], "Bahrain": ["bahrain", "manama"], "Kuwait": ["kuwait"],
    "Oman": ["oman", "muscat"], "Yemen": ["yemen", "sanaa", "aden"],
    "Iraq": ["iraq", "baghdad", "mosul", "najaf", "karbala", "basra", "erbil"],
    "Syria": ["syria", "damascus", "aleppo", "palmyra"],
    "Lebanon": ["leban", "beirut", "byblos", "baalbek"], "Jordan": ["jordan", "amman", "petra"],
    "Palestine": ["palestin", "gaza", "jerusalem", "ramallah", "bethlehem"],
    "Morocco": ["morocc", "marrakech", "fez", "casablanca", "rabat"],
    "Algeria": ["alger", "algiers"], "Tunisia": ["tunis"], "Libya": ["libya", "tripoli", "benghazi"],
    "Sudan": ["sudan", "khartoum"],
}
def guess_countries(text):
    t = text.lower()
    return [c for c, keys in PLACES.items() if any(re.search(r"\b" + re.escape(k), t) for k in keys)]

dev = df[df["split"] == "dev"].copy()
if len(dev):
    dev["countries_named"] = (dev["question"] + " " + dev["answer"]).map(guess_countries)
    named = Counter(c for cs in dev["countries_named"] for c in cs)
    print(f"dev: {(dev['countries_named'].str.len() > 0).mean():.0%} of examples name a country or city")
    display(pd.Series(named).sort_values(ascending=False).to_frame("dev examples"))
    print(f"examples where a place is named only in the answer, not the question: "
          f"{sum(bool(set(guess_countries(a)) - set(guess_countries(q))) for q, a in zip(dev.question, dev.answer))}")

## 4. Save stats

In [ ]:
stats = {
    "rows": df.groupby("split").size().to_dict(),
    "answer_words": df.groupby("split")["a_words"].describe().round(1).to_dict(orient="index"),
    "style": style.groupby("split").mean().round(3).to_dict(orient="index"),
}
if not labeled.empty:
    stats["country"] = labeled["country"].value_counts().to_dict()
    stats["category"] = labeled["category"].value_counts().to_dict()
RESULTS_DIR.mkdir(exist_ok=True)
pd.Series(stats).to_json(RESULTS_DIR / "data_stats.json", indent=2)
print("saved", RESULTS_DIR / "data_stats.json")

## 5. Findings (dev, 1000 examples)

- **Gold answers are full sentences, not short phrases.** Median 24 words (5–95%: 13–41), 76% are one sentence, 23% two. Almost all start with "The …" (94%) and end with a period.
- **Answers restate the question.** About 40% of the question's content words come back in the answer ("What airline …?" → "The airplane represents Qatar Airways."). 15% start with "The image …", 26% refer to the image in some way.
- **Hedging is normal:** 25% use "likely", "may", "suggests" etc. Answers often add a line of cultural context after the fact.
- **Questions are almost all "What …?"** (99%): "What is", "What type", "What architectural", "What cultural".
- **Dev has no country/category labels.** A third of dev examples name a place, and in most of them (278 of ~330) the place appears **only in the answer**: the model has to recognise the country from the image. Egypt, Qatar, Morocco, UAE, Palestine, Saudi Arabia lead. Real distribution: from train/devtest.

**What this means for T2_Text:**

- BERTScore F1 against ~25-word references punishes one-word answers (low recall). The prompt should ask for **one or two full sentences, ~20–30 words**, starting by restating the subject ("The dish is …"), and **name the specific entity** (landmark, dish, airline, city).
- Next: check with `bertscore()` on the dev subset how short / medium / long answers score against gold (e.g. "Qatar Airways" vs "The airplane represents Qatar Airways.").